In [ ]:
# @title 1. build monkeytype (~ 2 minutes)
THEME = "chaos_theory"
REF = "938027d078fbe3d83d3f5c64e71f6172dda311d3"


import os, subprocess

subprocess.run(["bash", "-ec", r'''
R=/content
command -v npm >/dev/null || apt-get install -y -qq npm
mkdir -p $R/node && cd $R/node
[ -e node_modules/.bin/pnpm ] || npm i --prefix . node@24 pnpm@11.21.0 --no-audit --no-fund --loglevel=error
export PATH=$R/node/node_modules/.bin:$PATH HUSKY=0 CI=1
node -v
mkdir -p $R/mt && cd $R/mt
[ -d .git ] || git init -q
git fetch -q --depth 1 https://github.com/monkeytypegame/monkeytype "$REF"
git checkout -q -f FETCH_HEAD
pnpm install --frozen-lockfile --filter "@monkeytype/frontend..." --reporter=append-only
C=frontend/src/ts/constants
# no Firebase project -> dummy config so the app boots logged-out
sed 's/###FIREBASE_APIKEY###/offline/;s/###FIREBASE_AUTHDOMAIN###/localhost/;s/###FIREBASE_PROJECTID###/offline/;s/###FIREBASE_STORAGEBUCKET###/offline.appspot.com/;s/###FIREBASE_MESSAGINGSENDERID###/0/;s/###FIREBASE_APPID###/1:0:web:0/' docker/frontend/firebase-config-live.ts > $C/firebase-config-live.ts
cp $C/firebase-config-live.ts $C/firebase-config.ts
grep -Eq "^  \"?$THEME\"?: \{" $C/themes.ts || { echo "unknown theme: $THEME"; exit 1; }
R=frontend/src/ts/router/create-router.ts
# inline mode has no real page URL: router keeps its own in-memory history when __MT_INLINE is set (normal builds unaffected)
sed -i 's/^  createBrowserHistory,$/  createBrowserHistory,\n  createMemoryHistory,/' $R
sed -i 's#const history = options.history ?? createBrowserHistory();#const history = options.history ?? ((globalThis as any).__MT_INLINE ? createMemoryHistory({ initialEntries: ["/"] }) : createBrowserHistory());#' $R
grep -q __MT_INLINE $R && grep -q "^  createMemoryHistory," $R
cd frontend
RECAPTCHA_SITE_KEY=6LeIxAcTAAAAAJcZVRqyHh71UMIEGNQ_MXjiZKhI BACKEND_URL=/api pnpm exec vite build
# first-visit theme: seed saved config (same path as a returning user's saved theme)
sed -i "s#</head>#<script>try{localStorage.getItem('config')||localStorage.setItem('config',JSON.stringify({theme:'$THEME'}))}catch(e){}</script></head>#" dist/index.html
grep -q "setItem('config'" dist/index.html
'''], check=True, env={**os.environ, "THEME": THEME, "REF": REF})
print("built ->", "/content/mt/frontend/dist")


In [ ]:
# @title 2. run and have fun ;)
HEIGHT = 800

import base64, glob, gzip, json, os, re, subprocess
from IPython.display import HTML, display

D = "/content/mt/frontend/dist"
MIME = {"json": "application/json", "css": "text/css", "woff2": "font/woff2", "png": "image/png", "webp": "image/webp", "jpg": "image/jpeg", "svg": "image/svg+xml"}
b64 = lambda b: base64.b64encode(b).decode()
rd = lambda p: open(os.path.join(D, p.lstrip("/")), "rb").read()
mime = lambda p: MIME[p.rsplit(".", 1)[-1]]

def datauri(m):
    p = m.group(1)
    ok = os.path.isfile(os.path.join(D, p.lstrip("/"))) and (not p.startswith("/webfonts/") or re.match(r"/webfonts/(RobotoMono|LexendDeca|fa-|Roboto-Regular|Oxygen-Regular|Ubuntu-Regular)", p))
    return f"url(data:{mime(p)};base64,{b64(rd(p))})" if ok else m.group(0)

def css(p):
    return re.sub(r"""url\(\s*["']?(/(?:images|webfonts)/[^"')]+)["']?\s*\)""", datauri, rd(p).decode()).encode()

page_html = rd("index.html").decode()
entry = re.search(r'<script type="module"[^>]*src="/(js/[^"]+)"', page_html).group(1)
eb = glob.glob("/content/mt/node_modules/.pnpm/esbuild@*/node_modules/esbuild/bin/esbuild")[-1]
env = {**os.environ, "PATH": "/content/node/node_modules/.bin:" + os.environ["PATH"]}
subprocess.run([eb, entry, "--bundle", "--format=esm", "--target=esnext", "--outfile=/tmp/app.js", "--log-level=error"], cwd=D, env=env, check=True)

files = {"/app.js": ("text/javascript", open("/tmp/app.js", "rb").read()),
         "/app.css": ("text/css", css(glob.glob(D + "/css/monkeytype.*.css")[0][len(D):]))}
for pat in ["quotes/english.json", "layouts/qwerty.json", "languages/english*.json", "themes/*.css", "funbox/*.css"]:
    for f in sorted(glob.glob(os.path.join(D, pat))):
        p = f[len(D):]
        if p.startswith("/languages/") and not re.fullmatch(r"/languages/english(_1k|_5k|_10k|_25k)?\.json", p): continue
        files[p] = (mime(p), css(p) if p.endswith(".css") else rd(p))
pack = {p: [m, b64(gzip.compress(b, 9))] for p, (m, b) in files.items()}

head = page_html.split("</head>")[0]
heads = [[m.group(1), m.group(2)] for m in re.finditer(r"<(style|script)(?![^>]*\bsrc=)[^>]*>(.*?)</\1>", head, re.S)]

BOOT = r"""
(async () => {
  const P = __PACK__, HEAD = __HEAD__, cache = {};
  const dec = async s => new Uint8Array(await new Response(new Blob([Uint8Array.from(atob(s), c => c.charCodeAt(0))]).stream().pipeThrough(new DecompressionStream('gzip'))).arrayBuffer());
  const get = p => (cache[p] ??= P[p] ? dec(P[p][1]) : Promise.resolve(null));
  const txt = async p => { const b = await get(p); return b ? new TextDecoder().decode(b) : null; };
  const rf = window.fetch.bind(window);
  const ASSET = /(?:^|\/)((?:quotes|languages|layouts|themes|funbox|challenges|sounds|images|webfonts|webfonts-preview)\/.+|(?:version|contributors|supporters)\.json)$/;
  window.fetch = async (input, init) => {  // serve the site's static files from memory; no backend
    let u; try { u = new URL(typeof input === 'string' ? input : input.url, location.href); } catch (e) { return rf(input, init); }
    if (u.origin !== location.origin) return rf(input, init);
    if (u.pathname.startsWith('/api/')) return new Response('{"message":"offline","data":null}', {status: 404, headers: {'Content-Type': 'application/json'}});
    const m = u.pathname.match(ASSET);  // relative urls resolve against the frame's own path, so match by tail
    if (!m) return rf(input, init);
    const p = '/' + m[1], b = await get(p);
    return b ? new Response(b, {status: 200, headers: {'Content-Type': P[p][0]}}) : new Response('', {status: 404});
  };
  // theme / funbox <link href=/themes/x.css>: apply as inline <style>, fire load like a real stylesheet would
  const SH = /^\/?(themes|funbox)\/[^?#]+\.css$/;
  const virt = async (l, path) => {
    const c = await txt(path), s = l.__s || (l.__s = document.createElement('style'));
    s.textContent = c || '';
    if (!s.isConnected) l.parentNode ? l.after(s) : document.head.appendChild(s);
    l.dispatchEvent(new Event(c == null ? 'error' : 'load'));
  };
  const sa = Element.prototype.setAttribute;
  Element.prototype.setAttribute = function (n, v) {
    if (this instanceof HTMLLinkElement && n === 'href') {
      v = String(v);
      if (SH.test(v)) { virt(this, '/' + v.replace(/^\//, '')); return; }
      if (/^\/js\/.+\.js$/.test(v)) return;  // modulepreload hints: bundle is already inline
      if (this.__s) this.__s.textContent = '';
    }
    return sa.call(this, n, v);
  };
  const hd = Object.getOwnPropertyDescriptor(HTMLLinkElement.prototype, 'href');
  Object.defineProperty(HTMLLinkElement.prototype, 'href', {configurable: true, get: hd.get, set(v) { this.setAttribute('href', v); }});
  new MutationObserver(ms => ms.forEach(m => m.removedNodes.forEach(n => n.__s && n.__s.remove()))).observe(document, {subtree: true, childList: true});

  try { localStorage.getItem('x'); } catch (e) {  // storage blocked in this frame: keep settings in memory instead of crashing
    const m = new Map(), f = {getItem: k => m.has(k) ? m.get(k) : null, setItem: (k, v) => void m.set(k, String(v)), removeItem: k => void m.delete(k), clear: () => m.clear(), key: i => [...m.keys()][i] ?? null, get length() { return m.size; }};
    for (const k of ['localStorage', 'sessionStorage']) Object.defineProperty(window, k, {value: f, configurable: true});
  }
  for (const [t, body] of HEAD) { const e = document.createElement(t); e.textContent = body; document.head.appendChild(e); }
  const pad = document.createElement('style'); pad.textContent = 'body{min-height:__H__px}'; document.head.appendChild(pad);
  window.__MT_INLINE = true;  // router uses in-memory history (frame URL is not the site's URL)
  const st = document.createElement('style'); st.textContent = await txt('/app.css'); document.head.appendChild(st);
  const s = document.createElement('script'); s.type = 'module'; s.textContent = await txt('/app.js'); document.head.appendChild(s);
})();
""".replace("__PACK__", json.dumps(pack)).replace("__HEAD__", json.dumps(heads)).replace("__H__", str(HEIGHT))

page = f"<script>{BOOT}</script>"
display(HTML(page))
